# 05 - Inference & Sentence Recognition Pipeline

This notebook tests live character recognition and demonstrates the **Sentence & Word Recognition pipeline**:
1. Loading the trained 47-class CNN model and label mapping.
2. Testing individual character predictions on numbers and letters from `dataset/samples/`.
3. Demonstrating character segmentation on multi-character words and sentences.
4. Detecting word spaces based on character spacing thresholds.
5. Reconstructing the full text sentence string with bounding box annotations.


In [ ]:
# Import libraries
import os
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sys.path.append(os.path.abspath(".."))
from src.preprocessing import preprocess_single_image, load_label_mapping
from src.models import load_trained_model
from src.sentence_recognition import recognize_sentence, create_sample_sentence_image

print("Inference and sentence recognition modules loaded successfully!")


## 1. Load Model and Class Mapping


In [ ]:
model_path = os.path.abspath("../models/handwritten_cnn.keras")
mapping_path = os.path.abspath("../models/label_mapping.json")

model = load_trained_model(model_path)
label_map = load_label_mapping(mapping_path)
print(f"Loaded model supporting {len(label_map)} character classes.")


## 2. Test Single Character Recognition (Numbers & Letters)


In [ ]:
samples_dir = os.path.abspath("../dataset/samples")
sample_files = sorted(glob.glob(os.path.join(samples_dir, "sample_*.png")))
# Exclude sentence sample
char_samples = [p for p in sample_files if not p.endswith("sample_sentence.png")][:10]

fig, axes = plt.subplots(2, 5, figsize=(16, 7))

for ax, filepath in zip(axes.flat, char_samples):
    tensor, processed_pil = preprocess_single_image(filepath)
    probs = model.predict(tensor, verbose=0)[0]
    pred_class = np.argmax(probs)
    confidence = probs[pred_class] * 100
    char_label = label_map.get(pred_class, str(pred_class))
    
    ax.imshow(processed_pil, cmap="gray")
    filename = os.path.basename(filepath)
    ax.set_title(f"{filename}\nPred: '{char_label}' ({confidence:.1f}%)", fontweight="bold", fontsize=11)
    ax.axis("off")

plt.suptitle("Inference on Sample Handwritten Numbers and Letters", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()


## 3. End-to-End Sentence Recognition
Testing sentence segmentation, character classification, and text reconstruction on a handwritten sentence.


In [ ]:
# Create or load sample sentence image ('AI 2026')
sample_sentence_path = os.path.abspath("../dataset/samples/sample_sentence.png")
if not os.path.exists(sample_sentence_path):
    create_sample_sentence_image("AI 2026", samples_dir=samples_dir, output_path=sample_sentence_path)

# Recognize sentence
recognized_text, char_details, annotated_img = recognize_sentence(
    sample_sentence_path, model, label_map
)

print(f"\nRecognized Sentence: '{recognized_text}'")
print("-" * 50)
for i, d in enumerate(char_details, 1):
    print(f"Char {i}: '{d['character']}' | Confidence: {d['confidence']:.2f}% | BBox: {d['bbox']} | Space: {d['space_after']}")

# Display annotated segmentation
plt.figure(figsize=(10, 4))
plt.imshow(annotated_img)
plt.title(f"Recognized Sentence: '{recognized_text}'", fontsize=14, fontweight="bold")
plt.axis("off")
plt.show()


## Summary & Next Steps
- The model successfully recognizes both numbers (0–9) and letters (A–Z).
- Sentence recognition accurately segments characters, detects spaces, and reconstructs full sentences.
- Launch the interactive Streamlit app to test single characters and multi-word sentences:
  ```bash
  streamlit run app.py
  ```
